# Hubble-8B unlearning — NPO

Trains **NPO** from the Perturbed 8B checkpoint, pushes the model to HF, evaluates infill accuracy, and uploads results. Run the cells top to bottom on an **A100 (80GB)**.

> All four method notebooks are independent; the **master notebook** aggregates their HF results.

In [1]:
# --- Setup: install deps, put the hubble8b package on the path, set HF token ---
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.53", "datasets", "accelerate", "bitsandbytes", "scipy"], check=False)

# Make `import hubble8b` work. Adjust HUB_DIR to wherever this repo lives on the box.
HUB_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
for cand in [HUB_DIR, os.path.join(HUB_DIR, "gpu_8b_run"), "/kaggle/input"]:
    if os.path.isdir(os.path.join(cand, "hubble8b")):
        sys.path.insert(0, cand)
        break

# HF token: use env in batch jobs; Colab userdata only when available.
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    except Exception:
        pass

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is not set. Export HF_TOKEN before submitting the job; nbconvert cannot prompt interactively.")

import hubble8b as h8
h8.hf_login()
print("device:", h8.DEVICE, "| target slot:", h8.TARGET_SLOT, "| LSR slot:", h8.LSR_SLOT)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[hf] logged in.
device: cuda:0 | target slot: birthdate | LSR slot: occupation


In [2]:
# --- Data: splits + retain + distractor pool (deterministic, seed=42) ---
forget_set, keep_set, unseen_set = h8.load_yago_splits()
retain = h8.load_retain_texts()
retain_texts = retain.select(range(min(len(retain), len(forget_set) * 40)))["text"]
splits = {"Unseen": unseen_set, "Unlearn": forget_set, "Keep": keep_set}
import json
pool_birth = h8.build_distractor_pool(forget_set, keep_set, unseen_set, h8.TARGET_SLOT)
print("birthdate distractors:", len(pool_birth))


[data] Unlearn=44 Keep=45 Unseen=250


[data] retain (WikiText): 797211 rows
birthdate distractors: 334


In [3]:
MODEL_NAME = "NPO"

In [4]:
# --- Train NPO (inits from Perturbed; 8B needs A100 80GB) ---
import pandas as pd
model, hist = h8.TRAINERS["npo"](forget_set["text"], retain_texts)
h8.push_model(model, h8.get_tokenizer(), "npo")                 # -> HF model repo
h8.upload_artifact(  # training curve
    (lambda p: (pd.DataFrame(hist).to_csv(p, index=False) or p))(
        f"{h8.workdir()}/npo_train_history.csv"))


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

/home/harshparikh/gpu_8b_run/hubble8b/methods.py:67: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  hist.append(dict(step=step, forget_loss=float(forget_loss), retain_loss=float(retain_loss)))


[NPO] step 10 forget=0.2798 retain=3.2835 ratio=-42.624


[NPO] step 20 forget=0.0168 retain=2.1320 ratio=-70.836


... (4 outputs left out)

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


[hf] pushed model -> https://huggingface.co/Harsh01012/hubble-8b-npo-unlearned-yago-birthdate


[hf] uploaded npo_train_history.csv -> datasets/Harsh01012/hubble-8b-unlearning-results


In [5]:
# --- Evaluate this model: infill accuracy + push CSV to HF ---
tok = h8.get_tokenizer()
name = MODEL_NAME
df = h8.run_infill(model, name, splits, pool_birth, target_slot=h8.TARGET_SLOT)
import pandas as pd
p = f"{h8.workdir()}/infill_{name}.csv"; df.to_csv(p, index=False); h8.upload_artifact(p)
h8.to_cpu(model); del model; h8.free_cuda()
print(name, "done — infill CSV uploaded to HF.")


  [NPO] Unseen/name_only: acc=0.0840


  [NPO] Unseen/intro_prefix: acc=0.1240


  [NPO] Unseen/full_prefix: acc=0.1160


  [NPO] Unseen/full_prefix_full_suffix: acc=0.1280


  [NPO] Unlearn/name_only: acc=0.0682


  [NPO] Unlearn/intro_prefix: acc=0.0682


... (4 outputs left out)

  [NPO] Keep/full_prefix: acc=0.0889


  [NPO] Keep/full_prefix_full_suffix: acc=0.0667


[hf] uploaded infill_NPO.csv -> datasets/Harsh01012/hubble-8b-unlearning-results


NPO done — infill CSV uploaded to HF.
